In [ ]:
# %% [markdown]
# # Lung Atlas Integration Benchmark
# This notebook orchestrates scVI, SysVI, and scPoli across HVG subsets, collects embeddings, and benchmarks with scIB.

# %%
import os
import subprocess
import pandas as pd
import scanpy as sc
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import numpy as np

# -------------------------
# User parameters
# -------------------------
hvgs = [500, 1000, 2000]
methods = ["scvi", "sysvi", "scpoli"]
adata_dir = "adata_hvg_subsets"  # directory with preprocessed HVG files
file_dir = "results"             # directory to save embeddings
model_dir = "models"             # directory to save models
key_batch = ["condition", "batch"]   # adjust to your obs keys
latent_dims = 20
embedding_dims = 20
key_save = f"{'_'.join(key_batch)}_latent{latent_dims}_embed{embedding_dims}"

os.makedirs(file_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# Master AnnData
adata = sc.read_h5ad("adata_full_reference.h5ad")

# -------------------------
# Step 1: Run subprocesses
# -------------------------
for n in hvgs:
    adata_file = os.path.join(adata_dir, f"adata_{n}_hvg.h5ad")
    
    # Run scVI
    subprocess.run([
        "conda", "run", "-n", "env_new", "python", "run_scvi.py",
        n, adata_file, model_dir, file_dir, key_batch[0], key_batch[1], key_save, latent_dims, embedding_dims
    ])
    
    # Run SysVI
    subprocess.run([
        "conda", "run", "-n", "env_new", "python", "run_sysvi.py",
        n, adata_file, model_dir, file_dir, key_batch[0], key_batch[1], key_save, latent_dims, embedding_dims
    ])
    
    # Run scPoli
    subprocess.run([
        "conda", "run", "-n", "env_old", "python", "run_scpoli.py",
        n, adata_file, model_dir, file_dir, key_batch[1], key_save, latent_dims, embedding_dims
    ])

# -------------------------
# Step 2: Load embeddings into adata
# -------------------------
for method in methods:
    for n in hvgs:
        df_path = os.path.join(file_dir, f"{method}_latent_embd_{n}_{key_save}.csv")
        if not os.path.exists(df_path):
            print(f"⚠️ Missing embedding: {df_path}")
            continue
        
        latent_df = pd.read_csv(df_path, index_col=0)
        latent_df = latent_df.loc[adata.obs_names]  # ensure correct cell order
        adata.obsm[f"{method}_{n}"] = latent_df.to_numpy()

# -------------------------
# Step 3: Run scIB Benchmark
# -------------------------
bm = Benchmarker(
    adata,
    batch_key="batch",
    label_key="cell_type",
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
    embedding_obsm_keys=[f"{m}_{n}" for m in methods for n in hvgs],
    n_jobs=6
)

bm.benchmark()

# -------------------------
# Step 4: Visualize & save results
# -------------------------
bm.plot_results_table()
results = bm.get_results(min_max_scale=False)
results.to_csv(os.path.join(file_dir, "benchmark_results.csv"))

# Optional: transpose for easier viewing
results.transpose()
